## 01 - Bronze Ingestion
- Load data into Volumne to Delta table 
- `bronze_financial_data : the excel file that manually input

In [0]:
CATALOG = 'credit_risk_project'
VOLUME_PATH = f"/Volumes/{CATALOG}/credit_risk_project_bronze/bronze_raw_files"

REGISTRY_PATH = f"{VOLUME_PATH}/inpi"

## 1. Bronze - Company REgistry (json file from INPI)
**1 json file (folder), 1 company**

In [0]:
df_registry = (
    spark.read
    .option("multiLine", True)
    .json(f"{REGISTRY_PATH}/*/*.json")
)

In [0]:
print(f"Number of loaded: {df_registry.count()}")
df_registry.printSchema()

write Delta table  on raw (No flatten sturct) keep nested

In [0]:
(
    df_registry.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{CATALOG}.credit_risk_project_bronze.bronze_company_registry")
)

print("Saved in bronze_company_registry")

# 2. Bronze - Financial Data (CSV file)

In [0]:
VOLUME_PATH_CSV = f"/Volumes/{CATALOG}/credit_risk_project_bronze/"

FINANCIAL_CSV_PATH = f"{VOLUME_PATH_CSV}/bronze_raw_files/financial_data_extracted.csv"

df_financial = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FINANCIAL_CSV_PATH)
)

print(f"Number of loaded: {df_financial.count()}")
df_financial.printSchema()


In [0]:
(
    df_financial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{CATALOG}.credit_risk_project_bronze.bronze_financial_data")
)

print("Saved in bronze_financial_data")

## Check the result

In [0]:
display(spark.sql(
    """SELECT siren, formality.content.personneMorale.identite.entreprise.denomination AS company_name 
    FROM credit_risk_project.credit_risk_project_bronze.bronze_company_registry"""
))
    
# df_registry.select(
#     "siren",
#     "formality.content.personneMorale.identite.entreprise.denomination"
# ).show()

In [0]:
display(spark.sql(
    """SELECT SIREN, Company_Name 
    FROM credit_risk_project.credit_risk_project_bronze.bronze_financial_data"""
))